# Implementação do Multi-Layer Perceptron em R

In [89]:
# Iniciar o Colab com R
# https://colab.research.google.com/#create=true&language=r



Usando a função logistica como função de ativação

Razões:

1. Introduz não-linearidade
2. Produz saída entre 0 e 1 (probabilidade)
3. Tem derivada simples, facilitando o backpropagation

In [90]:
#a função de ativação 
funcao.ativacao <- function(v){
    #funcao logistica
    y <- 1/(1+exp(-v))

    return(y)
}

In [91]:
#implementando a derivada da função logistica
der.funcao.ativacao <- function(y){
    derivada <- y * (1-y)

    return(derivada)
}

In [92]:
#criando uma arquitetura para a MLP
arquitetura <- function(num.entrada, num.escondida, num.saida,
                        funcao.ativacao, der.funcao.ativacao) {
    arq <- list()

    #parametro da rede
    arq$num.entrada <- num.entrada
    arq$num.escondida <- num.escondida
    arq$num.saida <- num.saida
    arq$funcao.ativacao <- funcao.ativacao
    arq$der.funcao.ativacao <- der.funcao.ativacao

    # 2 neuronios na camada escondida
    #
    #       Ent1    Ent2   Bias
    # 1     w11     w12    w13
    # 2     w21     w22    w23

    # Pesos conectando entrada e escondida
    #adicionamos +1 por conta dos pesos do bias
    num.pesos.entrada.escondida <- (num.entrada+1)*num.escondida
    arq$escondida <- matrix(runif(min=-0.5, max=0.5, num.pesos.entrada.escondida),
                                nrow = num.escondida, ncol=num.entrada+1)

    #Pesos conectando escondida e saida
    num.pesos.escondida.saida <- (num.escondida+1)*num.saida
    arq$saida <- matrix(runif(min=-0.5,max=0.5, num.pesos.escondida.saida),
                            nrow = num.saida, ncol = num.escondida+1)

    return(arq)

}

fase de propagação

In [93]:
mlp.propagacao <- function(arq, dados.exemplos){

    #entrada -> camada escondida
    #multiplicacao de matriz : %*%
    v.entrada.escondida <- arq$escondida %*% as.numeric(c(dados.exemplos, 1))
    y.entrada.escondida <- arq$funcao.ativacao(v.entrada.escondida)

    #camada escondida -> camada de saida
    v.escondida.saida <- arq$saida %*% c(y.entrada.escondida, 1)
    y.escondida.saida <- arq$funcao.ativacao(v.escondida.saida)

    #resultados
    resultados <- list()
    resultados$v.entrada.escondida <- v.entrada.escondida
    resultados$y.entrada.escondida <- y.entrada.escondida
    resultados$v.escondida.saida <- v.escondida.saida
    resultados$y.escondida.saida <- y.escondida.saida

    return(resultados)
}

**Backpropagation**

 O que é: Um método supervisionado de aprendizado que utiliza o gradiente descendente para otimizar pesos e vieses (bias).

Objetivo: Minimizar a função de custo (erro) entre a saída prevista e a saída real.

Importância: Sem o backpropagation, redes neurais multicamadas não conseguiriam ajustar eficientemente as camadas ocultas

Funcionamento:

**Forward Pass (Propagação Direta)**: Dados de entrada passam pela rede, camadas ocultas processam e geram uma saída.

**Cálculo do Erro:** A saída da rede é comparada com a "verdade básica" (saída desejada), gerando uma função de perda.

**Backward Pass (Retropropagação):** O erro é propagado de volta. O algoritmo calcula a derivada parcial da função de custo em relação a cada peso, utilizando a regra da cadeia do cálculo.

**Atualização dos Pesos:** Os pesos são ajustados (subtraindo o gradiente multiplicado por uma taxa de aprendizado) para reduzir o erro na próxima iteração.

In [94]:
mlp.retropropagacao <- function(arq, dados, n, limiar){

    erroQuadratico <- 2 * limiar
    epocas <- 0

    #O treinamento ocorre enquanto o erroQuad > limiar
    while(erroQuadratico > limiar){
        erroQuadratico <- 0

        #Treino para todos os exemplos(i.e epoca)
        for(i in 1:nrow(dados)){
            #pega um exemplo de entrada
            x.entrada <- dados[i,1:arq$num.entrada]
            x.saida <- dados[i, ncol(dados)]

            #Foward Pass
            #pega a saida da rede para o exemplo
            result <- mlp.propagacao(arq, x.entrada)
            y <- result$y.escondida.saida

            #Calculo do erro
            erro <- x.saida - y

            #soma erro quadratico
            erroQuadratico <- erroQuadratico + erro*erro

            #Backward Pass

            # Gradiente local no neuronio de saida
            # erro * derivada da funcao de ativacao
            gradiente.local.saida <- erro * arq$der.funcao.ativacao(y)

            # Gradiente local no neuronio escondido
            # derivada da funcao de ativacao no neuronio escondido * soma dos gradientes
            # locais dos neuronios conectados na proxima camada * pesos conectando a camada
            # escondida com a saida
            pesos.saida <- arq$saida[,1:arq$num.escondida]
            gradiente.local.escondida <- as.numeric(arq$der.funcao.ativacao(result$y.entrada.escondida)) * (gradiente.local.saida %*% pesos.saida)

            #Atualizando os pesos sinapticos
            arq$saida <- arq$saida + n * (gradiente.local.saida %*% c(result$y.entrada.escondida, 1))

            #escondida
            arq$escondida <- arq$escondida + n * (t(gradiente.local.escondida) %*% as.numeric(c(x.entrada,1)))
        }

        erroQuadratico <- erroQuadratico / nrow(dados)
        cat("Erro Quadratico Medio = ", erroQuadratico, "\n")
        flush.console()
        epocas <- epocas+1
    }

    retorno <- list()
    retorno$arq <- arq
    retorno$epocas <- epocas

    return(retorno)
}

In [95]:
# Leitura de uma tabela com os dados do problema XOR
dados <- read.table('XOR.txt')
print(dados)

  V1 V2 V3
1  0  0  0
2  0  1  1
3  1  0  1
4  1  1  0


In [96]:
# Nossa arquitetura
arq <- arquitetura(2,2,1,funcao.ativacao, der.funcao.ativacao)
print(arq)

$num.entrada
[1] 2

$num.escondida
[1] 2

$num.saida
[1] 1

$funcao.ativacao
<srcref: file "" chars 2:20 to 7:1>

$der.funcao.ativacao
<srcref: file "" chars 2:24 to 6:1>

$escondida
            [,1]        [,2]        [,3]
[1,] 0.049755519  0.41944757  0.14931495
[2,] 0.007575612 -0.04143384 -0.09152398

$saida
           [,1]       [,2]       [,3]
[1,] -0.1885314 -0.2036894 -0.4211724



In [97]:
# Vamos treinar nossa MLP
#taxa de aprendizado: 0.1
#limiar: 0.001
modelo <- mlp.retropropagacao(arq,dados,0.1,1e-3)
print(modelo)

Erro Quadratico Medio =  0.2747255 
Erro Quadratico Medio =  0.2733142 
Erro Quadratico Medio =  0.2719775 
Erro Quadratico Medio =  0.2707132 
Erro Quadratico Medio =  0.2695191 
Erro Quadratico Medio =  0.2683927 
Erro Quadratico Medio =  0.2673315 
Erro Quadratico Medio =  0.2663329 
Erro Quadratico Medio =  0.2653942 
Erro Quadratico Medio =  0.2645128 
Erro Quadratico Medio =  0.263686 
Erro Quadratico Medio =  0.2629111 
Erro Quadratico Medio =  0.2621855 
Erro Quadratico Medio =  0.2615067 
Erro Quadratico Medio =  0.260872 
Erro Quadratico Medio =  0.2602791 
Erro Quadratico Medio =  0.2597256 
Erro Quadratico Medio =  0.2592092 
Erro Quadratico Medio =  0.2587278 
Erro Quadratico Medio =  0.2582791 
Erro Quadratico Medio =  0.2578612 
Erro Quadratico Medio =  0.2574721 
Erro Quadratico Medio =  0.2571101 
Erro Quadratico Medio =  0.2567734 
Erro Quadratico Medio =  0.2564603 
Erro Quadratico Medio =  0.2561694 
Erro Quadratico Medio =  0.255899 
Erro Quadratico Medio =  0.2556

In [98]:
#Testando em dados novos
retorno <- mlp.propagacao(modelo$arq,c(0,1))
print(retorno$y.escondida.saida)

print(mlp.propagacao(modelo$arq,c(1,0))$y.escondida.saida)

print(mlp.propagacao(modelo$arq,c(1,1))$y.escondida.saida)

print(mlp.propagacao(modelo$arq,c(0,0))$y.escondida.saida)

          [,1]
[1,] 0.9699309
          [,1]
[1,] 0.9699388
           [,1]
[1,] 0.03766261
           [,1]
[1,] 0.02778792


In [ ]:
# Leitura de dados sobre risco de crédito.
# Queremos prever, com base nas variáveis de entrada, se ocorrerá ou não um calote em 10 anos

dataset <- read.csv("creditset.csv")
dim(dataset)
head(dataset)

# clientid: identificação do cliente
# income: renda anual
# age: idade
# loan: empréstimo com data de no mínimo 10 anos atrás
# LTI: razão entre valor do empréstimo e renda anual
# default10yr: se ocorrerá (1) ou não (0) um calote em 10 anos

In [ ]:
# Vamos utilizar como variáveis de entrada LTI e age
dados <- dataset[,c(3,5,6)]
dados[,1:2] <- scale(dados[,1:2]) # normalização dos dados (media 0 e desvio 1)
head(dados)

In [ ]:
# Vamos escolher aleatoriamente dados para treino e teste.
# O conjunto de dados já está randomizado.
# Assim, Vamos pegar os primeiros 1400 exemplos para treino e o restante para teste
dados.treino <- dados[1:1400,]
dados.teste <- dados[1401:2000,]

In [ ]:
# Vamos utilizar como variáveis de entrada LTI e age
dados <- dataset[,c(3,5,6)]
dados[,1:2] <- scale(dados[,1:2]) # normalização dos dados (media 0 e desvio 1)
head(dados)

In [ ]:
# Vamos treinar nossa rede com 4 neurônios na camada escondida
modelo <- mlp.retropropagacao(arq,dados.treino,0.3,1e-3)
print(modelo)

In [ ]:
# Fazendo predicoes para cada exemplo de teste
predicoes <- vector()
for(i in 1:nrow(dados.teste)){

  pred <- mlp.propagacao(modelo$arq,dados.teste[i,1:2])$y.escondida.saida

  predicoes <- c(predicoes,pred)
}
print(predicoes)

In [ ]:
# Criando uma matriz para comparação dos resultados
matriz.comparacao <- cbind(dados.teste[,3],predicoes)
colnames(matriz.comparacao) <- c('V','P')
print(matriz.comparacao)

In [ ]:
# Matriz de confusão com o arredondamento das predições
mc <- table(matriz.comparacao[,1],round(matriz.comparacao[,2]))
print(mc)

acc <- sum(diag(mc))/sum(mc)
print(acc)
sum(mc)
sum(diag(mc))